# 03 - Evaluate (Phase 6 in Fabric)
Loads the answer key and runs the key evaluation queries in Spark SQL. The same questions in T-SQL (for the Lakehouse SQL analytics endpoint or a Warehouse) are in `fabric/sql/metrics_queries_tsql.sql`.

In [ ]:
def csv_to_table(path, table):
    df = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    df.write.mode("overwrite").format("delta").saveAsTable(table)

csv_to_table("Files/data/ground_truth/enquiries_truth.csv", "ground_truth")
csv_to_table("Files/data/ground_truth/raw_row_map.csv", "raw_row_map")

### Overall accuracy and urgent cases missed

In [ ]:
display(spark.sql('''
SELECT COUNT(*) AS enquiries,
       SUM(CASE WHEN t.category = g.category THEN 1 ELSE 0 END) AS category_right,
       SUM(CASE WHEN t.urgency  = g.urgency  THEN 1 ELSE 0 END) AS urgency_right
FROM raw_row_map m
JOIN ground_truth g ON g.enquiry_id = m.enquiry_id
JOIN ai_triage t    ON t.raw_row = CAST(m.raw_row AS INT)
WHERE m.row_type = 'original'
'''))

In [ ]:
display(spark.sql('''
SELECT COUNT(*) AS truly_urgent,
       SUM(CASE WHEN t.urgency = 'High' THEN 1 ELSE 0 END) AS ai_caught,
       SUM(CASE WHEN t.urgency IS NULL OR t.urgency <> 'High' THEN 1 ELSE 0 END) AS missed_by_ai,
       SUM(CASE WHEN (t.urgency IS NULL OR t.urgency <> 'High') AND t.needs_human_review = 0 THEN 1 ELSE 0 END) AS missed_and_not_reviewed
FROM raw_row_map m
JOIN ground_truth g ON g.enquiry_id = m.enquiry_id
JOIN ai_triage t    ON t.raw_row = CAST(m.raw_row AS INT)
WHERE m.row_type = 'original' AND g.urgency = 'High'
'''))

### Then: Power BI
In the Lakehouse, choose **New semantic model** (select the tables above), then build the report from `powerbi/BUILD_GUIDE.md`. The `human_reviews` step runs locally in this project; in Fabric you would add a Data Pipeline plus a Power Automate or n8n approval step.